# 시드 고정

In [1]:
import os
import random
import numpy as np
import torch
import pandas as pd


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(1)

# 데이터 처리

### 전역 설정
- 잠자는 시간, 깨어있는 시간, 일 하는 시간, 자유시간
- 공휴일

In [2]:
SLEEP_HOURS = tuple(range(0, 7))
ACTIVE_HOURS = tuple(range(7, 24))
WORK_HOURS = tuple(range(7, 19))
FREE_HOURS = tuple(range(19, 24))

HOLIDAY_DATES = [
    pd.Timestamp('2024-08-15'),
    pd.Timestamp('2024-09-16'),
    pd.Timestamp('2024-09-17'),
    pd.Timestamp('2024-09-18'),
    pd.Timestamp('2024-10-03'),
    pd.Timestamp('2024-10-09'),
]

### 유틸 함수

In [3]:
from pathlib import Path


DATA_DIR = Path("./data")

In [4]:
from enum import Enum


class DataType(Enum):
    mACStatus = "mACStatus"
    mActivity = "mActivity"
    mAmbience = "mAmbience"
    mBle = "mBle"
    mGps = "mGps"
    mLight = "mLight"
    mScreenStatus = "mScreenStatus"
    mUsageStats = "mUsageStats"
    mWifi = "mWifi"
    wHr = "wHr"
    wLight = "wLight"
    wPedo = "wPedo"

In [5]:
import pandas as pd


def load_data(data_type: DataType):
    file_path = DATA_DIR / f"ch2025_data_items/ch2025_{data_type.value}.parquet"
    df = pd.read_parquet(file_path)
    df["subject_id"] = df["subject_id"].astype("category")
    df["lifelog_date"] = df["timestamp"].dt.normalize()
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute
    df["weekday"] = df["timestamp"].dt.weekday

    fixed_columns = ["subject_id", "timestamp", "lifelog_date", "month", "day", "hour", "minute", "weekday"]
    columns = df.columns.tolist()
    columns = fixed_columns + [col for col in columns if col not in fixed_columns]
    df = df[columns]
    df = df.sort_values(by=["subject_id", "timestamp"])

    return df


def load_train():
    df = pd.read_csv(DATA_DIR / "ch2025_metrics_train.csv")
    df["subject_id"] = df["subject_id"].astype("category")
    df["sleep_date"] = pd.to_datetime(df["sleep_date"]).dt.normalize()
    df["lifelog_date"] = pd.to_datetime(df["lifelog_date"]).dt.normalize()
    return df


def load_val():
    from io import StringIO
    train_df = load_train()
    val_ids = "subject_id,sleep_date\nid01,2024-07-24\nid01,2024-07-27\nid01,2024-08-18\nid01,2024-08-19\nid01,2024-08-20\nid01,2024-08-21\nid01,2024-08-22\nid01,2024-08-24\nid01,2024-08-25\nid01,2024-08-26\nid01,2024-08-27\nid01,2024-08-28\nid01,2024-08-29\nid01,2024-08-30\nid02,2024-08-23\nid02,2024-08-24\nid02,2024-09-16\nid02,2024-09-17\nid02,2024-09-19\nid02,2024-09-20\nid02,2024-09-21\nid02,2024-09-22\nid02,2024-09-23\nid02,2024-09-24\nid02,2024-09-25\nid02,2024-09-26\nid02,2024-09-27\nid02,2024-09-28\nid03,2024-08-30\nid03,2024-09-01\nid03,2024-09-02\nid03,2024-09-03\nid03,2024-09-05\nid03,2024-09-06\nid03,2024-09-07\nid04,2024-09-03\nid04,2024-09-04\nid04,2024-09-05\nid04,2024-09-06\nid04,2024-09-07\nid04,2024-09-08\nid04,2024-09-09\nid04,2024-10-08\nid04,2024-10-09\nid04,2024-10-10\nid04,2024-10-11\nid04,2024-10-12\nid04,2024-10-13\nid04,2024-10-14\nid05,2024-10-19\nid05,2024-10-23\nid05,2024-10-24\nid05,2024-10-25\nid05,2024-10-26\nid05,2024-10-27\nid05,2024-10-28\nid06,2024-07-25\nid06,2024-07-26\nid06,2024-07-27\nid06,2024-07-28\nid06,2024-07-29\nid06,2024-07-30\nid06,2024-07-31\nid07,2024-07-07\nid07,2024-07-08\nid07,2024-07-09\nid07,2024-07-10\nid07,2024-07-11\nid07,2024-07-12\nid07,2024-07-13\nid07,2024-07-30\nid07,2024-08-01\nid07,2024-08-02\nid07,2024-08-03\nid07,2024-08-04\nid07,2024-08-05\nid07,2024-08-06\nid08,2024-08-28\nid08,2024-08-29\nid08,2024-08-30\nid08,2024-08-31\nid08,2024-09-01\nid08,2024-09-02\nid08,2024-09-04\nid09,2024-08-02\nid09,2024-08-22\nid09,2024-08-23\nid09,2024-08-24\nid09,2024-08-25\nid09,2024-08-27\nid09,2024-08-28\nid09,2024-08-29\nid09,2024-08-30\nid09,2024-08-31\nid09,2024-09-01\nid09,2024-09-02\nid09,2024-09-03\nid09,2024-09-04\nid10,2024-08-28\nid10,2024-08-30\nid10,2024-08-31\nid10,2024-09-01\nid10,2024-09-02\nid10,2024-09-03\nid10,2024-09-06\n"
    val_df = pd.read_csv(StringIO(val_ids))
    val_df = val_df.astype({"subject_id": "category", "sleep_date": "datetime64[ns]"})
    val_df = train_df.merge(val_df, on=["subject_id", "sleep_date"], how="inner")
    return val_df


def load_test():
    df = pd.read_csv(DATA_DIR / "ch2025_submission_sample.csv")
    df["subject_id"] = df["subject_id"].astype("category")
    df["sleep_date"] = pd.to_datetime(df["sleep_date"]).dt.normalize()
    df["lifelog_date"] = pd.to_datetime(df["lifelog_date"]).dt.normalize()
    return df


In [6]:
def shift_lifelog_date(df, target_hours=SLEEP_HOURS):
    df = df.copy()
    mask = df["hour"].isin(target_hours) & df["hour"].lt(12)
    df.loc[mask, "lifelog_date"] = df.loc[mask, "lifelog_date"] - pd.Timedelta(days=1)
    df.loc[mask, "day"] = df.loc[mask, "day"] - 1
    df = df.sort_values(by=["subject_id", "lifelog_date", "timestamp"])
    return df

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', lambda x: '%0.4f' % x)

def describe_df(df):
    print(f"# shape:\n{df.shape}\n")
    print(f"# dtypes:\n{df.dtypes}\n")
    print(f"# head:\n{df.head(3)}\n")
    nan_stats = df.isna().sum().to_frame(name='missing_count')
    nan_stats['missing_ratio(%)'] = (df.isna().mean() * 100).round(2)
    print(f"# nan_stats:\n" + nan_stats.to_string() + "\n")

### ✔️ mACStatus 핸드폰 충전상태
- Indicates whether the smartphone is currently being charged.
- m_charging : 0/1 상태
- 핸드폰이 오랫 동안 충전했다는 의미?
 - 한 자리에 장시간 머물러 있었다.
 - 핸드폰을 장시간 사용하지 않았다.  

In [8]:
mACStatus_ori = load_data(DataType.mACStatus)
mACStatus_ori = shift_lifelog_date(mACStatus_ori, target_hours=SLEEP_HOURS)

len(mACStatus_ori), mACStatus_ori.head(1)

(939896,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  m_charging
 0       id01 2024-06-26 12:03:00   2024-06-26      6   26    12       3        2           0)

In [9]:
def run_length_encoding(arr):
    """Run-Length Encoding"""
    if len(arr) == 0:
        return []

    diffs = np.diff(np.concatenate(([0], arr, [0])))
    run_starts = np.where(diffs == 1)[0]
    run_ends = np.where(diffs == -1)[0]
    return run_ends - run_starts

def process_mACStatus(df):
    status = df["m_charging"].values

    def _process_feature(status):
        if len(status) == 0:
            return 0., 0., 0., 0., 0.

        # charging 상태 비율, 합
        ratio_charging = status.mean()
        sum_charging = status.sum()

        # 상태전이 횟수
        transitions = (status[1:] != status[:-1]).sum()

        lengths = run_length_encoding(status)
        avg_charging_duration = np.mean(lengths) if len(lengths) > 0 else 0
        max_charging_duration = np.max(lengths) if len(lengths) > 0 else 0

        return ratio_charging, sum_charging, transitions, avg_charging_duration, max_charging_duration

    # 하루
    charging_ratio, charging_sum, chargning_transitions, avg_charging_duration, max_charging_duration = _process_feature(status)

    # 잠자는 시간대
    sleep_status = status[df["hour"].isin(SLEEP_HOURS)]
    sleep_charging_ratio, sleep_charging_sum, sleep_charging_transitions, sleep_avg_charging_duration, sleep_max_charging_duration = _process_feature(sleep_status)

    return pd.Series({
        'charging_ratio': charging_ratio,
        'charging_sum': charging_sum,
        'charging_transitions': chargning_transitions,
        'avg_charging_duration': avg_charging_duration,
        'max_charging_duration': max_charging_duration,
        'sleep_charging_ratio': sleep_charging_ratio,
        'sleep_charging_sum': sleep_charging_sum,
        'sleep_charging_transitions': sleep_charging_transitions,
        'sleep_avg_charging_duration': sleep_avg_charging_duration,
        'sleep_max_charging_duration': sleep_max_charging_duration,
    })

mACStatus = (
    mACStatus_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mACStatus)
    .reset_index(drop=True)
)
describe_df(mACStatus)

# shape:
(803, 12)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
charging_ratio                        float64
charging_sum                          float64
charging_transitions                  float64
avg_charging_duration                 float64
max_charging_duration                 float64
sleep_charging_ratio                  float64
sleep_charging_sum                    float64
sleep_charging_transitions            float64
sleep_avg_charging_duration           float64
sleep_max_charging_duration           float64
dtype: object

# head:
  subject_id lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration
0       id01   2024-06-26          0.1626      179.0000               31.0000                11.1875                41.0000                0.07

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\216347306.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mACStatus)


### ✔️ mActivity 추정행동
- Value calculated by the Google Activity Recognition API.
 - 0 : IN_VEHICLE
 - 1 : ON_BICYCLE
 - 2 : ON_FOOT
 - 3 : STILL (not moving)
 - 4 : UNKNOWN
 - 5 : TILTING (This often occurs when a device is picked up from a desk or a user who is sitting stands up.)
 - 7 : WALKING
 - 8 : RUNNING
- 근무시간   : 오전 7시부터 오후 6시까지
- 근무외시간 : 오후6시부터 12시까지

In [10]:
mActivity_ori = load_data(DataType.mActivity)
mActivity_ori = shift_lifelog_date(mActivity_ori, target_hours=SLEEP_HOURS)

len(mActivity_ori),mActivity_ori.head(1)

(961062,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  m_activity
 0       id01 2024-06-26 12:03:00   2024-06-26      6   26    12       3        2           4)

In [11]:
def process_mActivity(df):
    activity = df["m_activity"].values.astype("int8")

    EXCLUDE_ACTIVITY = [3, 4]
    WALKING_ACTIVITY = [1, 2, 7, 8]
    VEHICLE_ACTIVITY = [0]

    def _process_feature(activity):
        if len(activity) == 0:
            return 0., 0., 0.
        
        # Walking minutes
        walking_minutes = np.isin(activity, WALKING_ACTIVITY).sum()

        # Vehicle minutes
        vehicle_minutes = np.isin(activity, VEHICLE_ACTIVITY).sum()

        # Activity minutes
        activity_minutes = (1 - np.isin(activity, EXCLUDE_ACTIVITY)).sum()

        return walking_minutes, vehicle_minutes, activity_minutes
        
    # 하루
    walking_minutes, vehicle_minutes, activity_minutes = _process_feature(activity)

    # 잠자는 시간대
    sleep_walking_minutes, sleep_vehicle_minutes, sleep_activity_minutes = _process_feature(activity[df["hour"].isin(SLEEP_HOURS)])

    
    return pd.Series({
        'walking_minutes': walking_minutes,
        'vehicle_minutes': vehicle_minutes,
        'activity_minutes': activity_minutes,
        'sleep_walking_minutes': sleep_walking_minutes,
        'sleep_vehicle_minutes': sleep_vehicle_minutes,
        'sleep_activity_minutes': sleep_activity_minutes,
    })

mActivity = (
    mActivity_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mActivity)
    .reset_index(drop=True)
)
describe_df(mActivity)

# shape:
(803, 8)

# dtypes:
subject_id                      category
lifelog_date              datetime64[ns]
walking_minutes                  float64
vehicle_minutes                  float64
activity_minutes                 float64
sleep_walking_minutes            float64
sleep_vehicle_minutes            float64
sleep_activity_minutes           float64
dtype: object

# head:
  subject_id lifelog_date  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes
0       id01   2024-06-26          37.0000         116.0000          153.0000                 5.0000                27.0000                 32.0000
1       id01   2024-06-27          30.0000         213.0000          243.0000                 4.0000                29.0000                 33.0000
2       id01   2024-06-28          36.0000         145.0000          181.0000                 3.0000                13.0000                 16.0000

# nan_stats:
              

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\2047524889.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mActivity)


### ✔️ mAmbience 추정주변소리
- Ambient sound identification labels and their respective probabilities.
- 무슨 소리가 난게 중요할까?
- 새벽에 무슨 소리던지 소리가 난게 중요한 걸까?
- 여러 가지 소리 중에 노이즈도 포함되어 있을까?

In [12]:
mAmbience_ori = load_data(DataType.mAmbience)
mAmbience_ori = shift_lifelog_date(mAmbience_ori, target_hours=SLEEP_HOURS)

len(mAmbience_ori), mAmbience_ori.head(1)

(476577,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                   m_ambience
 0       id01 2024-06-26 13:00:10   2024-06-26      6   26    13       0        2  [[Music, 0.30902618], [Vehicle, 0.081680894], [Motor vehicle (road), 0.04035286], [Outside, urban or manmade, 0.037144363], [Outside, rural or natural, 0.032663062], [Car, 0.03199804], [Speech, 0.029806137], [Inside, large room or hall, 0.01684492], [Truck, 0.016206821], [Sound effect, 0.01591479]])

In [13]:
def process_mAmbience(df):
    ambience = df["m_ambience"].values  # [[label, prob], ...], [[label, prob], ...]

    def _process_feature(ambience):
        labels = []

        for amb in ambience:
            labels_, _ = zip(*amb)
            labels.extend(labels_)
        
        unique_label_count = len(labels)
        snor_count = len(list(filter(lambda x: "snor" in x.lower(), labels)))

        return unique_label_count, snor_count
    
    # 활동시간
    active_hour_unique_label_count, active_hour_snor_count = _process_feature(ambience[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는시간
    sleep_hour_unique_label_count, sleep_hour_snor_count = _process_feature(ambience[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_unique_label_count': active_hour_unique_label_count,
        'active_hour_snor_count': active_hour_snor_count,
        'sleep_hour_unique_label_count': sleep_hour_unique_label_count,
        'sleep_hour_snor_count': sleep_hour_snor_count,
    })


mAmbience = (
    mAmbience_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mAmbience)
    .reset_index(drop=True)
)
describe_df(mAmbience)

# shape:
(803, 6)

# dtypes:
subject_id                              category
lifelog_date                      datetime64[ns]
active_hour_unique_label_count             int64
active_hour_snor_count                     int64
sleep_hour_unique_label_count              int64
sleep_hour_snor_count                      int64
dtype: object

# head:
  subject_id lifelog_date  active_hour_unique_label_count  active_hour_snor_count  sleep_hour_unique_label_count  sleep_hour_snor_count
0       id01   2024-06-26                            3040                       4                           2100                      0
1       id01   2024-06-27                            5100                       0                           2100                      0
2       id01   2024-06-28                            5000                       0                           2100                      0

# nan_stats:
                                missing_count  missing_ratio(%)
subject_id                      

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\1264552170.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mAmbience)


### ✔️ mBle 블루투스
- Bluetooth devices around individual subject.
 - 7936 : Wearable, Headset, AV Device
 - 1796 : Peripheral (입력장치) 계열
 - 0 : 정보 없음 또는 알 수 없음(Unknown)
 - 1084 : Audio/Video (스피커, 헤드셋, 이어폰, TV 등)
 - 524 : Phone (휴대폰, 스마트폰)
 - 1060 : Headphones
 - 284 : commputer (PC, 노트북, PDA)

In [14]:
mBle_ori = load_data(DataType.mBle)
mBle_ori = shift_lifelog_date(mBle_ori, target_hours=SLEEP_HOURS)

len(mBle_ori), mBle_ori.head(1)

(21830,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [15]:
def process_mBle(df):
    ble = df["m_ble"].values  # [[{"address": "xx:xx:xx:xx:xx:xx", "device_class": "0", "rssi": -70}, ...], [...], ...]

    def _process_feature(ble):
        if len(ble) == 0:
            return 0., 0., 0., 0., 0.
        
        rssi = []
        devices = []
        for ble_data in ble:
            for device in ble_data:
                rssi.append(device["rssi"])
                devices.append(device["device_class"])

        rssi = np.array(rssi)
        rssi_mean = rssi.mean() if len(rssi) > 0 else 0
        rssi_min = rssi.min() if len(rssi) > 0 else 0
        rssi_max = rssi.max() if len(rssi) > 0 else 0

        unknown_count = devices.count("0")
        others_count = len(devices) - unknown_count
        others_ratio = others_count / len(devices) if len(devices) > 0 else 0
        unknown_ratio = unknown_count / len(devices) if len(devices) > 0 else 0

        return rssi_mean, rssi_min, rssi_max, others_ratio, unknown_ratio
    
    # 일할때
    work_hour_rssi_mean, work_hour_rssi_min, work_hour_rssi_max, work_hour_others_ratio, work_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(WORK_HOURS)])

    # 퇴근후
    free_hour_rssi_mean, free_hour_rssi_min, free_hour_rssi_max, free_hour_others_ratio, free_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(FREE_HOURS)])

    # 잠자는시간
    sleep_hour_rssi_mean, sleep_hour_rssi_min, sleep_hour_rssi_max, sleep_hour_others_ratio, sleep_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'work_hour_rssi_mean': work_hour_rssi_mean,
        'work_hour_rssi_min': work_hour_rssi_min,
        'work_hour_rssi_max': work_hour_rssi_max,
        'work_hour_others_ratio': work_hour_others_ratio,
        'work_hour_unknown_ratio': work_hour_unknown_ratio,
        'free_hour_rssi_mean': free_hour_rssi_mean,
        'free_hour_rssi_min': free_hour_rssi_min,
        'free_hour_rssi_max': free_hour_rssi_max,
        'free_hour_others_ratio': free_hour_others_ratio,
        'free_hour_unknown_ratio': free_hour_unknown_ratio,
        'sleep_hour_rssi_mean': sleep_hour_rssi_mean,
        'sleep_hour_rssi_min': sleep_hour_rssi_min,
        'sleep_hour_rssi_max': sleep_hour_rssi_max,
        'sleep_hour_others_ratio': sleep_hour_others_ratio,
        'sleep_hour_unknown_ratio': sleep_hour_unknown_ratio
    })
mBle = (
    mBle_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mBle)
    .reset_index(drop=True)
)
describe_df(mBle)


# shape:
(726, 17)

# dtypes:
subject_id                        category
lifelog_date                datetime64[ns]
work_hour_rssi_mean                float64
work_hour_rssi_min                 float64
work_hour_rssi_max                 float64
work_hour_others_ratio             float64
work_hour_unknown_ratio            float64
free_hour_rssi_mean                float64
free_hour_rssi_min                 float64
free_hour_rssi_max                 float64
free_hour_others_ratio             float64
free_hour_unknown_ratio            float64
sleep_hour_rssi_mean               float64
sleep_hour_rssi_min                float64
sleep_hour_rssi_max                float64
sleep_hour_others_ratio            float64
sleep_hour_unknown_ratio           float64
dtype: object

# head:
  subject_id lifelog_date  work_hour_rssi_mean  work_hour_rssi_min  work_hour_rssi_max  work_hour_others_ratio  work_hour_unknown_ratio  free_hour_rssi_mean  free_hour_rssi_min  free_hour_rssi_max  free_hour_others_r

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\4233196545.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mBle)


### ✔️ mGps, GPS 기반 핸드폰 위치
- Multiple GPS coordinates measured within a single minute using the smartphone.
- speed가 1보다 큰경우 정지 상태가 아니고 움직이고 있다고 판단
 - 0.5-2 : 걸어서 이동하는 경우  
 - 2-5 : 조깅
 - 5 이상 : 차를 타고 이동하는 경우

- speed가 0.5-2사이를 하루에 몇분동안 지속했는지?
- speed가 2-5사이를 하루에 몇분동안 지속했는지? (유산소 운동 시간)
- speed가 5이상을 하루에 몇분동안 지속했는지?  

In [16]:
mGps_ori = load_data(DataType.mGps)
mGps_ori = shift_lifelog_date(mGps_ori, target_hours=SLEEP_HOURS)

len(mGps_ori), mGps_ori.head(1)

(800611,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [17]:
def haversine_np(lon1, lat1, lon2, lat2, radius=6371):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return radius * c

def process_mGps(df):
    gps = df["m_gps"].values  # [[{'altitude': 110.6, 'latitude': 0.2077385, 'longitude': 0.170027, 'speed': 0.0}, ...], ...]
    timestamps = df["timestamp"].values

    def _process_feature(gps, timestamps):
        if len(gps) == 0:
            return 0., 0., 0., 0., 0., 0., 0.

        # n-분 단위
        latitudes = []
        longitudes = []
        altitudes = []
        speeds = []
        minutes = []  # 누적 분

        for i, (gps_data, timestamp) in enumerate(zip(gps, timestamps)):
            _latitudes = []
            _longitudes = []
            _altitudes = []
            _speeds = []
            for data in gps_data:
                _latitudes.append(data["latitude"])
                _longitudes.append(data["longitude"])
                _altitudes.append(data["altitude"])
                _speeds.append(data["speed"])
            
            latitudes.append(np.mean(_latitudes))
            longitudes.append(np.mean(_longitudes))
            altitudes.append(np.mean(_altitudes))
            speeds.append(np.mean(_speeds))
            minutes.append(1 if i == 0 else pd.Timedelta(timestamps[i] - timestamps[i-1]).total_seconds() / 60)

        latitudes = np.array(latitudes)
        longitudes = np.array(longitudes)
        altitudes = np.array(altitudes)
        speeds = np.array(speeds)
        minutes = np.array(minutes)

        walk_minutes = minutes[(speeds >= 0.5) & (speeds < 2.0)].sum()
        jog_minutes = minutes[(2.0 <= speeds) & (speeds < 5.0)].sum()
        vehicle_minutes = minutes[(5.0 <= speeds)].sum()

        # 속도
        mean_speed = speeds.mean() if len(speeds) > 0 else 0
        max_speed = speeds.max() if len(speeds) > 0 else 0
        min_speed = speeds.min() if len(speeds) > 0 else 0

        # 이동거리
        distance = haversine_np(longitudes[:-1], latitudes[:-1], longitudes[1:], latitudes[1:]).sum()

        return walk_minutes, jog_minutes, vehicle_minutes, mean_speed, max_speed, min_speed, distance

    # 하루
    active_hour_walk_minutes, active_hour_jog_minutes, active_hour_vehicle_minutes, active_hour_mean_speed, active_hour_max_speed, active_hour_min_speed, active_hour_distance = _process_feature(gps[df["hour"].isin(ACTIVE_HOURS)], timestamps[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_walk_minutes, sleep_hour_jog_minutes, sleep_hour_vehicle_minutes, sleep_hour_mean_speed, sleep_hour_max_speed, sleep_hour_min_speed, sleep_hour_distance = _process_feature(gps[df["hour"].isin(SLEEP_HOURS)], timestamps[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_walk_minutes': active_hour_walk_minutes,
        'active_hour_jog_minutes': active_hour_jog_minutes,
        'active_hour_vehicle_minutes': active_hour_vehicle_minutes,
        'active_hour_mean_speed': active_hour_mean_speed,
        'active_hour_max_speed': active_hour_max_speed,
        'active_hour_min_speed': active_hour_min_speed,
        'active_hour_distance': active_hour_distance,
        'exercise_flag': 1 if active_hour_jog_minutes > 10 else 0,  # n분 이상 조깅한 경우
        'sleep_hour_walk_minutes': sleep_hour_walk_minutes,
        'sleep_hour_jog_minutes': sleep_hour_jog_minutes,
        'sleep_hour_vehicle_minutes': sleep_hour_vehicle_minutes,
        'sleep_hour_mean_speed': sleep_hour_mean_speed,
        'sleep_hour_max_speed': sleep_hour_max_speed,
        'sleep_hour_min_speed': sleep_hour_min_speed,
        'sleep_hour_distance': sleep_hour_distance
    })

mGps = (
    mGps_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mGps)
    .reset_index(drop=True)
)
describe_df(mGps)


# shape:
(759, 17)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
active_hour_walk_minutes              float64
active_hour_jog_minutes               float64
active_hour_vehicle_minutes           float64
active_hour_mean_speed                float64
active_hour_max_speed                 float64
active_hour_min_speed                 float64
active_hour_distance                  float64
exercise_flag                         float64
sleep_hour_walk_minutes               float64
sleep_hour_jog_minutes                float64
sleep_hour_vehicle_minutes            float64
sleep_hour_mean_speed                 float64
sleep_hour_max_speed                  float64
sleep_hour_min_speed                  float64
sleep_hour_distance                   float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_walk_minutes  active_hour_jog_minutes  active_hour_vehicle_minutes  active_hour_mean_speed  active_hour_max_speed  active_hour

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\757796829.py:91: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mGps)


### 🔥 mLight 주변 밝기 (TODO)
- Ambient light measured by the smartphone.
 - 어두운 밤	0.1 ~ 1 lux	캄캄한 방, 달빛 없는 밤
 - 가로등 켜진 거리	10 ~ 20 lux	흐릿한 외부 조명
 - 실내 조명	100 ~ 500 lux	사무실, 일반 거실
 - 밝은 실외	10,000 ~ 25,000 lux	맑은 날 햇빛
 - 직사광선 아래	30,000 ~ 100,000 lux	여름 한낮, 매우 강한 햇빛

- 밝기에 따라서 언제 불을 끄고 잠든 시간 추정
- 직사광선 잠에 좋은 영향을 주는지? (논문)
- 결측치 처리 x -> 제외하고 평균

In [18]:
mLight_ori = load_data(DataType.mLight)
mLight_ori = shift_lifelog_date(mLight_ori, target_hours=SLEEP_HOURS)
len(mLight_ori), mLight_ori.head(1)

(96258,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  m_light
 0       id01 2024-06-26 12:03:00   2024-06-26      6   26    12       3        2 534.0000)

In [19]:
def process_mLight(df):
    light = df["m_light"].values  # [534.0, 224, ...]

    def _process_feature(light):
        if len(light) == 0:
            return 0., 0., 0., 0.

        ligths = np.array(light)
        mean_light = ligths.mean() if len(ligths) > 0 else 0
        min_light = ligths.min() if len(ligths) > 0 else 0
        max_light = ligths.max() if len(ligths) > 0 else 0
        std_light = ligths.std() if len(ligths) > 0 else 0

        return mean_light, min_light, max_light, std_light
    
    # 하루
    active_hour_mean_light, active_hour_min_light, active_hour_max_light, active_hour_std_light = _process_feature(light[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_mean_light, sleep_hour_min_light, sleep_hour_max_light, sleep_hour_std_light = _process_feature(light[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_mean_light': active_hour_mean_light,
        'active_hour_min_light': active_hour_min_light,
        'active_hour_max_light': active_hour_max_light,
        'active_hour_std_light': active_hour_std_light,
        'sleep_hour_mean_light': sleep_hour_mean_light,
        'sleep_hour_min_light': sleep_hour_min_light,
        'sleep_hour_max_light': sleep_hour_max_light,
        'sleep_hour_std_light': sleep_hour_std_light
    })

mLight = (
    mLight_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mLight)
    .reset_index(drop=True)
)
describe_df(mLight)

# shape:
(802, 10)

# dtypes:
subject_id                      category
lifelog_date              datetime64[ns]
active_hour_mean_light           float64
active_hour_min_light            float64
active_hour_max_light            float64
active_hour_std_light            float64
sleep_hour_mean_light            float64
sleep_hour_min_light             float64
sleep_hour_max_light             float64
sleep_hour_std_light             float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_mean_light  active_hour_min_light  active_hour_max_light  active_hour_std_light  sleep_hour_mean_light  sleep_hour_min_light  sleep_hour_max_light  sleep_hour_std_light
0       id01   2024-06-26                364.5068                 0.0000              1886.0000               392.9401                44.2619                0.0000              502.0000              127.2300
1       id01   2024-06-27                450.5784                 0.0000             11248.0000              1521.9885   

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\1628823064.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mLight)


### 🔥 mScreenStatus 화면 사용여부 (TODO)

- Indicates whether the smartphone screen is in use.
 - 기상시간, 취침시간, 수면시간
 - 휴대폰 이용횟수, 이용시간
 - 00 - 05 사이에 휴대폰 이용한 건수
 - 결측치 처리 x

In [20]:
mScreenStatus_ori = load_data(DataType.mScreenStatus)
mScreenStatus_ori = shift_lifelog_date(mScreenStatus_ori, target_hours=SLEEP_HOURS)
len(mScreenStatus_ori), mScreenStatus_ori.head(1)

(939653,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  m_screen_use
 0       id01 2024-06-26 12:03:00   2024-06-26      6   26    12       3        2             0)

In [21]:
def process_mScreenStatus(df):
    screen_use = df["m_screen_use"].values  # [0, 1, 0, ...]
    screen_use = np.array(screen_use).astype("int8")

    def _process_feature(screen_use):
        if len(screen_use) == 0:
            return 0., 0., 0.

        screen_use = np.array(screen_use)
        screen_use_ratio = screen_use.mean() if len(screen_use) > 0 else 0
        screen_use_sum = screen_use.sum() if len(screen_use) > 0 else 0
        screen_use_transitions = (screen_use[1:] != screen_use[:-1]).sum()

        return screen_use_ratio, screen_use_sum, screen_use_transitions
    
    # 하루
    screen_use_ratio, screen_use_sum, screen_use_transitions = _process_feature(screen_use)

    # 잠자는 시간대
    sleep_screen_use_ratio, sleep_screen_use_sum, sleep_screen_use_transitions = _process_feature(screen_use[df["hour"].isin(SLEEP_HOURS)])
    return pd.Series({
        'screen_use_ratio': screen_use_ratio,
        'screen_use_sum': screen_use_sum,
        'screen_use_transitions': screen_use_transitions,
        'sleep_screen_use_ratio': sleep_screen_use_ratio,
        'sleep_screen_use_sum': sleep_screen_use_sum,
        'sleep_screen_use_transitions': sleep_screen_use_transitions
    })

mScreenStatus = (
    mScreenStatus_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mScreenStatus)
    .reset_index(drop=True)
)
describe_df(mScreenStatus)


# shape:
(803, 8)

# dtypes:
subject_id                            category
lifelog_date                    datetime64[ns]
screen_use_ratio                       float64
screen_use_sum                         float64
screen_use_transitions                 float64
sleep_screen_use_ratio                 float64
sleep_screen_use_sum                   float64
sleep_screen_use_transitions           float64
dtype: object

# head:
  subject_id lifelog_date  screen_use_ratio  screen_use_sum  screen_use_transitions  sleep_screen_use_ratio  sleep_screen_use_sum  sleep_screen_use_transitions
0       id01   2024-06-26            0.2177        244.0000                 61.0000                  0.0829               34.0000                        7.0000
1       id01   2024-06-27            0.3503        501.0000                 81.0000                  0.0293               12.0000                        6.0000
2       id01   2024-06-28            0.3140        449.0000                 87.0000         

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\517058994.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mScreenStatus)


### 🔥 mUsageStats 앱사용통계  (TODO)
- mUsageStats: Indicates which apps were used on the smartphone and for how long.

 - 몇시까지 핸드폰 보다가 잠잤는지
 - 통화, 전화 얼마나 했는지
 - YouTube 얼마나 봤는지
 - 메시지, 카카오톡 얼마나 했는지
 - NAVER 얼마나 했는지
 - 평소보다 얼마나 많은 앱을 이용했는지
 - 제외? -> 시스템 UI,One UI 홈

In [22]:
mUsageStats_ori = load_data(DataType.mUsageStats)
mUsageStats_ori = shift_lifelog_date(mUsageStats_ori, target_hours=SLEEP_HOURS)
len(mUsageStats_ori), mUsageStats_ori.head(1)

(45197,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                      m_usage_stats
 0       id01 2024-06-26 13:00:00   2024-06-26      6   26    13       0        2  [{'app_name': ' 캐시워크', 'total_time': 69}, {'app_name': 'NAVER', 'total_time': 549}, {'app_name': ' ✝️성경일독Q', 'total_time': 7337}])

In [23]:
app_names = set()
for app_list in mUsageStats_ori["m_usage_stats"].values:
    for app in app_list:
        app_names.add(app["app_name"])

app_names = sorted(list(app_names))
app_names[:3]

['(구)티머니onda', '(캐시아워)', '11번가']

In [24]:
# gpt 한테 카테고리 만들라고 시킴 (https://chatgpt.com/share/e/6825e3d0-4bc0-8009-98ad-ee3836bc0fd3)
app_category_to_names = { "금융": [ "11번가", "AIA생명", "IBK 기업은행", "KB Pay", "KB스타뱅킹", "KS Fit", "MG더뱅킹", "MG손해보험 다이렉트", "NH pay", "NH기업뱅킹", "NH뱅킹", "NH앱캐시", "NH콕뱅크", "OK Cashbag", "PASS", "PAYCO", "Samsung Wallet", "Syrup", "The건강보험", "Toss", "핀크", "하나머니", "하나은행", "하나카드", "한국투자", "한화손해보험", "현대카드", "흥국화재", "카카오페이", "케이뱅크", "토스", "우체국보험", "웰컴디지털뱅크", "카카오뱅크", "삼성카드", "삼성화재 다이렉트 착" ], "기타": [ "(구)티머니onda", "(캐시아워)", "AirVisual", "AlwaysOnDisplay", "Android 시스템", "AnyAUTH", "Arkcraft_v1", "Authentication Framework", "Auto Clicker", "Auto Redial", "Avis Corporate", "BNKR몰", "CHARGEV", "CNCITY에너지", "DRAWELY", "Expert RAW", "Gaming Hub", "Gateman", "Good Lock", "Google", "Google 음성 인식 및 합성", "H.Point", "Headphones", "Home", "IDF Mobilités", "IntentResolver", "K-패스", "KAIST IdCard", "LAVU", "LG ThinQ", "LH청약센터", "LIVE스코어", "Letter Fonts", "MTP 애플리케이션", "MY FANS", "MY네컷", "Maps & Navigation", "Master for Minecraft", "Mi Home", "Mobile HR", "MyKia", "NFC", "One UI 홈", "Pi", "Prime Ruler", "Quick Share", "RoomEstimateApp", "SIM 카드 툴킷", "Samsung Checkout", "Samsung Pass 자동 완성", "SecSoundPicker", "Secure SignIn", "Smart Home", "Smart Switch", "Smart View", "Smart​Things", "Start", "StudioMate", "Tasks", "T world", "TD infinite", "Tapo", "TasteBuds", "VpnDialogs", "WORKS", "Whiteout Survival", "Wi-Fi 연결 팁", "Windows와 연결", "Yodha Pro", "b.stage", "blind", "com.dreamsecurity.MobileRelay.SampleCrypto", "duit+", "help-CNUH24", "i-ONE 알림", "iM뱅크", "monimo", "vFlat Scan", "가족돌봄", "계산기", "고용24", "교육원 전자출결", "국세청 손택스", "굿웨어몰", "권한 관리자", "긴급 SOS", "데이터 복원 도구", "디바이스 케어", "디지털 웰빙", "땡큐캠핑", "똑똑계산기", "멀티 컨트롤", "미디어 선택 도구", "바로청구", "배경화면 및 스타일", "보안 폴더", "부속 기기 관리자", "비디오 플레이어", "빅스비 루틴", "빅스비 보이스", "빅스비 비전", "삼성 계정", "삼성 캡처", "삼성 클라우드", "삼성 키보드", "생체 인식", "설정", "시계", "시스템 UI", "시프티", "앱 소리 분리 재생", "어시스턴트", "연락처", "영상통화 효과", "오피넷", "주변 디바이스 찾기", "접근성", "추천 설정", "추천 앱", "출입예약시스템", "키 체인", "통화", "통화 설정", "패키지 설치 프로그램", "휴대폰분실보호", "히어로즈" ], "쇼핑": [ "11번가", "ABC-MART", "AliExpress", "Amway", "BHC", "BNKR몰", "CJ온스타일", "GS SHOP", "G마켓", "KREAM", "KT알파 쇼핑", "NS홈쇼핑", "Nike", "SHEIN", "SSG.COM", "Temu", "iHerb", "무신사", "다나와 가격비교", "다이소몰", "롯데ON", "롯데잇츠", "롯데홈쇼핑", "메가MGC커피", "메가박스", "버거킹", "보리보리", "신세계몰", "신세계쇼핑", "아이디어스", "에누리 가격비교", "에이블리", "엘포인트", "옥션", "올리브영", "컴포즈커피", "쿠우쿠우", "쿠쿠통합몰", "쿠팡", "쿠팡이츠", "쿠팡플레이", "퀸잇", "크몽", "하이버", "하프클럽", "홈플러스", "이마트24", "제주항공", "젤캔들샵", "카지노" ], "게임": [ "1945 Air Force", "Arkcraft_v1", "Art Puzzle", "Block Journey", "Block Puzzle", "Cake Sort", "Crossy Road", "Cytus II", "Darkness and Flame 1", "Darkness and Flame 2", "Darkness and Flame 3", "Darkness and Flame 4", "Domino Dreams", "FC Online M", "Fantastic Bricks", "Find Differences", "Find Out", "Friends Rush", "Goblins Wood", "Killer Sudoku", "Magic Tiles 3", "Merge Designer-Decor & Story", "Nonogram Elf", "Nonogram-Number games", "NumMatch", "Number Match", "Nuts & Bolts Jam", "Plague Inc.", "PokeRogue Offline", "Pokémon GO", "Pokémon UNITE", "Royal Match", "SimCity", "Steam", "Super Slime - Black Hole Game", "TFT", "Talking Tom Gold Run", "Whiteout Survival", "WoW 컴패니언", "눈을 떠요 야생소년", "매직 디펜스", "무한의 계단", "벽돌깨기 퀘스트", "세븐나이츠", "수확의 정석", "카트라이더 러쉬플러스", "클래시 로얄", "클래시오브클랜", "픽셀 포켓 모험", "퓨처파이트", "프렌즈팝콘" ], "여행/교통": [ "Android Auto", "Avenza Maps", "Booking.com", "Flightradar24", "Google Maps", "Grab", "Hotels.com", "IDF Mobilités", "Kia", "Kia Connect", "Kia Digital Key", "Omio", "Skyscanner", "TMAP", "Trip.com", "U+스마트홈", "Waze", "고속버스 티머니", "대전버스", "대전시 타슈(QR단말기전용)", "카카오내비", "카카오맵", "카카오버스", "코레일톡", "티머니GO", "티웨이항공" ], "건강": [ "GoFasting", "InBody", "KS Fit", "Samsung Health", "Withings" ], "음악": [ "FLO", "Melon", "YouTube Music", "무료음악 벨소리", "지니뮤직" ], "사진/영상": [ "AI Retouch - 객체 제거", "AR 이모지", "AR 이모지 스티커", "AR 이모지 에디터", "AR 존", "Adobe Acrobat", "B612", "CamScanner", "SNOW", "YouTube", "넷﻿플﻿릭﻿스", "TikTok", "TikTok-Lite" ], "소셜": [ "BAND", "Facebook", "Instagram", "LINE", "LIVE스코어", "Messenger", "Slack", "StarMaker", "Threads", "WeChat", "X", "blind", "리멤버" ], "생산성": [ "Drive", "Files by Google", "Gmail", "Microsoft 365 (Office)", "Microsoft Word", "Notion", "Outlook", "PowerPoint", "Samsung Notes", "TimeTree", "WPS Office", "Zoom", "네이버 MYBOX", "스프레드시트", "프레젠테이션" ], "유틸리티": [ "AhnLab V3 Mobile Plus", "AlwaysOnDisplay", "AnyAUTH", "Authenticator", "Edge", "ExpressVPN", "Files by Google", "Google Play 서비스", "Google Play 스토어", "Home", "Microsoft Launcher", "One UI 홈", "Quick Share", "Samsung Flow", "Samsung Free", "Samsung Members", "Samsung Pass", "Smart View" ], "교육": [ "갓피플성경", "개역개정 큰글성경", "성경", "항전ON", "합격 요양보호사" ], "식음료": [ "McDonald's", "SRT", "Starbucks", "배달의민족", "요기요", "엽기떡볶이", "투썸하트" ], "뉴스/정보": [ "Flightradar24", "네이트", "네이트메일", "재난문자" ], "라이프스타일": [ "Android 시스템", "LG ThinQ", "오늘의집", "헤이홈", "홈노크타운", "숲나들e", "지그재그", "핫핑", "샐러디", "아이쉐어링", "와디즈", "포인핸드" ] }
app_categories = list(app_category_to_names.keys())
app_name_to_category = {}
for category, names in app_category_to_names.items():
    for name in names:
        app_name_to_category[name] = category

In [25]:
app_categories

['금융',
 '기타',
 '쇼핑',
 '게임',
 '여행/교통',
 '건강',
 '음악',
 '사진/영상',
 '소셜',
 '생산성',
 '유틸리티',
 '교육',
 '식음료',
 '뉴스/정보',
 '라이프스타일']

In [26]:
def process_mUsageStats(df):
    usage_stats = df["m_usage_stats"].values  #  [[{'app_name': ' 캐시워크', 'total_time': 69}, {'app_name': 'NAVER', 'total_time': 549}, {'app_name': ' ✝️성경일독Q', 'total_time': 7337}], [...], ...] 

    def _process_feature(usage_stats):
        # 앱 카테고리별 사용시간
        category_usage_times = {
            category: 0. for category in app_categories
        }
        category_usage_times["UNKNOWN"] = 0.

        if len(usage_stats) == 0:
            return category_usage_times

        for usage_stat in usage_stats:
            for app in usage_stat:
                app_name = app["app_name"]
                useage_time = app["total_time"] * 0.001 / 60  # 밀리초 -> 초 -> 분

                category_usage_times[app_name_to_category.get(app_name, "UNKNOWN")] += useage_time
        
        return category_usage_times

    # 하루
    active_hour_category_usage_times = _process_feature(usage_stats[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_category_usage_times = _process_feature(usage_stats[df["hour"].isin(SLEEP_HOURS)])

    ret = {
        **{
            f"active_hour_{category}_usage_time": active_hour_category_usage_times[category]
            for category in app_categories
        },
        **{
            f"sleep_hour_{category}_usage_time": sleep_hour_category_usage_times[category]
            for category in app_categories
        }
    }

    return pd.Series(ret)

mUsageStats = (
    mUsageStats_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mUsageStats)
    .reset_index(drop=True)
)
describe_df(mUsageStats)


# shape:
(790, 32)

# dtypes:
subject_id                             category
lifelog_date                     datetime64[ns]
active_hour_금융_usage_time               float64
active_hour_기타_usage_time               float64
active_hour_쇼핑_usage_time               float64
active_hour_게임_usage_time               float64
active_hour_여행/교통_usage_time            float64
active_hour_건강_usage_time               float64
active_hour_음악_usage_time               float64
active_hour_사진/영상_usage_time            float64
active_hour_소셜_usage_time               float64
active_hour_생산성_usage_time              float64
active_hour_유틸리티_usage_time             float64
active_hour_교육_usage_time               float64
active_hour_식음료_usage_time              float64
active_hour_뉴스/정보_usage_time            float64
active_hour_라이프스타일_usage_time           float64
sleep_hour_금융_usage_time                float64
sleep_hour_기타_usage_time                float64
sleep_hour_쇼핑_usage_time                float64
sleep_hour

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\231677757.py:45: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mUsageStats)


### ✔️ mWifi 주변wifi 정보
- Wifi devices around individual subject.
 - -30 ~ -50 dBm	매우 강한 신호 (최적)
 - -51 ~ -60 dBm	강한 신호 (문제 없음)
 - -61 ~ -70 dBm	괜찮은 신호 (약간 느릴 수 있음)
 - -71 ~ -80 dBm	약한 신호 (끊김 주의)
 - -81 dBm 이하	매우 약한 신호 (거의 끊김)

In [27]:
mWifi_ori = load_data(DataType.mWifi)
mWifi_ori = shift_lifelog_date(mWifi_ori, target_hours=SLEEP_HOURS)
len(mWifi_ori), mWifi_ori.head(1)

(76336,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [28]:
def process_mWifi(df, threshold=-60):
    wifi = df["m_wifi"].values  # [ [{'bssid': 'a0:0f:37:9a:5d:8b', 'rssi': -78}, ...], ...]

    def _process_feature(wifi):
        if len(wifi) == 0:
            return 0, 0., 0.,

        bssids = set()
        rssis = []
        for wifi_data in wifi:
            for data in wifi_data:
                if data["rssi"] >= threshold:
                    bssids.add(data["bssid"])
                    rssis.append(data["rssi"])

        bssid_count = len(bssids)
        mean_rssi = np.mean(rssis) if len(rssis) > 0 else 0
        max_rssi = np.max(rssis) if len(rssis) > 0 else 0

        return bssid_count, mean_rssi, max_rssi
    
    # 하루
    active_hour_bssid_count, active_hour_mean_rssi, active_hour_max_rssi = _process_feature(wifi[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_bssid_count, sleep_hour_mean_rssi, sleep_hour_max_rssi = _process_feature(wifi[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_bssid_count': active_hour_bssid_count,
        'active_hour_mean_rssi': active_hour_mean_rssi,
        'active_hour_max_rssi': active_hour_max_rssi,
        'sleep_hour_bssid_count': sleep_hour_bssid_count,
        'sleep_hour_mean_rssi': sleep_hour_mean_rssi,
        'sleep_hour_max_rssi': sleep_hour_max_rssi
    })

mWifi = (
    mWifi_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mWifi)
    .reset_index(drop=True)
)
describe_df(mWifi)


# shape:
(785, 8)

# dtypes:
subject_id                       category
lifelog_date               datetime64[ns]
active_hour_bssid_count           float64
active_hour_mean_rssi             float64
active_hour_max_rssi              float64
sleep_hour_bssid_count            float64
sleep_hour_mean_rssi              float64
sleep_hour_max_rssi               float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_bssid_count  active_hour_mean_rssi  active_hour_max_rssi  sleep_hour_bssid_count  sleep_hour_mean_rssi  sleep_hour_max_rssi
0       id01   2024-06-26                  50.0000               -50.2712              -19.0000                 19.0000              -45.0116             -27.0000
1       id01   2024-06-27                  12.0000               -47.0130              -26.0000                 12.0000              -45.2979             -27.0000
2       id01   2024-06-28                  21.0000               -47.2000              -26.0000                 16.0000    

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\347450162.py:40: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mWifi)


### ✔️ wHr 심박동수
- Heart rate readings recorded by the smartwatch.


In [29]:
wHr_ori = load_data(DataType.wHr)
wHr_ori = shift_lifelog_date(wHr_ori, target_hours=SLEEP_HOURS)
len(wHr_ori), wHr_ori.head(1)

(382918,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                           heart_rate
 0       id01 2024-06-26 12:23:00   2024-06-26      6   26    12      23        2  [134, 134, 135, 133, 134, 135, 134, 135, 134, 133, 133, 133, 132, 132, 131, 131, 131, 132, 132, 134, 134, 134, 132, 130, 128, 126, 126, 126, 127, 129, 130, 129, 130, 130, 127, 127, 126, 125, 123])

In [30]:
def process_wHr(df):
    heart_rate = df["heart_rate"].values  # [[0, 1, 2, ...], ...]

    def _process_feature(heart_rate):
        if len(heart_rate) == 0:
            return 0., 0., 0., 0., 0.

        heart_rate = np.array(sum(map(lambda x: x.tolist(), heart_rate), []))
        mean_hr = heart_rate.mean() if len(heart_rate) > 0 else 0
        min_hr = heart_rate.min() if len(heart_rate) > 0 else 0
        max_hr = heart_rate.max() if len(heart_rate) > 0 else 0
        std_hr = heart_rate.std() if len(heart_rate) > 0 else 0
        high_hr = heart_rate[heart_rate > 100].sum()

        return mean_hr, min_hr, max_hr, std_hr, high_hr
    
    # 하루
    active_hour_mean_hr, active_hour_min_hr, active_hour_max_hr, active_hour_std_hr, active_hour_high_hr = _process_feature(heart_rate[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_mean_hr, sleep_hour_min_hr, sleep_hour_max_hr, sleep_hour_std_hr, sleep_hour_high_hr = _process_feature(heart_rate[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_mean_hr': active_hour_mean_hr,
        'active_hour_min_hr': active_hour_min_hr,
        'active_hour_max_hr': active_hour_max_hr,
        'active_hour_std_hr': active_hour_std_hr,
        'active_hour_high_hr': active_hour_high_hr,
        'sleep_hour_mean_hr': sleep_hour_mean_hr,
        'sleep_hour_min_hr': sleep_hour_min_hr,
        'sleep_hour_max_hr': sleep_hour_max_hr,
        'sleep_hour_std_hr': sleep_hour_std_hr,
        'sleep_hour_high_hr': sleep_hour_high_hr
    })

wHr = (
    wHr_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_wHr)
    .reset_index(drop=True)
)
describe_df(wHr)

# shape:
(686, 12)

# dtypes:
subject_id                   category
lifelog_date           datetime64[ns]
active_hour_mean_hr           float64
active_hour_min_hr            float64
active_hour_max_hr            float64
active_hour_std_hr            float64
active_hour_high_hr           float64
sleep_hour_mean_hr            float64
sleep_hour_min_hr             float64
sleep_hour_max_hr             float64
sleep_hour_std_hr             float64
sleep_hour_high_hr            float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_mean_hr  active_hour_min_hr  active_hour_max_hr  active_hour_std_hr  active_hour_high_hr  sleep_hour_mean_hr  sleep_hour_min_hr  sleep_hour_max_hr  sleep_hour_std_hr  sleep_hour_high_hr
0       id01   2024-06-26              81.2434             59.0000            142.0000             11.8712          243191.0000             79.4238            58.0000           113.0000            11.5325           9716.0000
1       id01   2024-06-27              79

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\67742303.py:39: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_wHr)


### ✔️ wLight 앰비언트 라이트
- Ambient light measured by the smartwatch.  
  - 어두운 밤 0.1 ~ 1 lux 캄캄한 방, 달빛 없는 밤
  - 가로등 켜진 거리 10 ~ 20 lux 흐릿한 외부 조명
  - 실내 조명 100 ~ 500 lux 사무실, 일반 거실
  - 밝은 실외 10,000 ~ 25,000 lux 맑은 날 햇빛
  - 직사광선 아래 30,000 ~ 100,000 lux 여름 한낮, 매우 강한 햇빛

In [31]:
wLight_ori = load_data(DataType.wLight)
wLight_ori = shift_lifelog_date(wLight_ori, target_hours=SLEEP_HOURS)
len(wLight_ori), wLight_ori.head(1)

(633741,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  w_light
 0       id01 2024-06-26 12:17:00   2024-06-26      6   26    12      17        2 633.0000)

In [32]:
def process_mLight(df):
    light = df["w_light"].values  # [534.0, 224, ...]

    def _process_feature(light):
        if len(light) == 0:
            return 0., 0., 0., 0.

        ligths = np.array(light)
        mean_light = ligths.mean() if len(ligths) > 0 else 0
        min_light = ligths.min() if len(ligths) > 0 else 0
        max_light = ligths.max() if len(ligths) > 0 else 0
        std_light = ligths.std() if len(ligths) > 0 else 0

        return mean_light, min_light, max_light, std_light
    
    # 하루
    active_hour_mean_light, active_hour_min_light, active_hour_max_light, active_hour_std_light = _process_feature(light[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_mean_light, sleep_hour_min_light, sleep_hour_max_light, sleep_hour_std_light = _process_feature(light[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_mean_light': active_hour_mean_light,
        'active_hour_min_light': active_hour_min_light,
        'active_hour_max_light': active_hour_max_light,
        'active_hour_std_light': active_hour_std_light,
        'sleep_hour_mean_light': sleep_hour_mean_light,
        'sleep_hour_min_light': sleep_hour_min_light,
        'sleep_hour_max_light': sleep_hour_max_light,
        'sleep_hour_std_light': sleep_hour_std_light
    })

wLight = (
    wLight_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mLight)
    .reset_index(drop=True)
)
describe_df(wLight)

# shape:
(750, 10)

# dtypes:
subject_id                      category
lifelog_date              datetime64[ns]
active_hour_mean_light           float64
active_hour_min_light            float64
active_hour_max_light            float64
active_hour_std_light            float64
sleep_hour_mean_light            float64
sleep_hour_min_light             float64
sleep_hour_max_light             float64
sleep_hour_std_light             float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_mean_light  active_hour_min_light  active_hour_max_light  active_hour_std_light  sleep_hour_mean_light  sleep_hour_min_light  sleep_hour_max_light  sleep_hour_std_light
0       id01   2024-06-26                299.4151                 0.0000             20874.0000              1220.1126                44.4110                0.0000              737.0000              114.9149
1       id01   2024-06-27                290.7522                 0.0000             12464.0000              1031.2252   

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\2814975233.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mLight)


### ✔️ wPedo 걸음수
- Step data recorded by the smartwatch.

In [33]:
wPedo_ori = load_data(DataType.wPedo)
wPedo_ori = shift_lifelog_date(wPedo_ori, target_hours=SLEEP_HOURS)
len(wPedo_ori), wPedo_ori.head(1)

(748100,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  step  step_frequency  running_step  walking_step  distance  speed  burned_calories
 0       id01 2024-06-26 12:09:00   2024-06-26      6   26    12       9        2    10          0.1667             0             0    8.3300 0.1388           0.0000)

In [34]:
def process_mPedo(df):

    def _process_feature(df):
        if len(df) == 0:
            return 0., 0., 0.

        steps = df["step"].values
        distances = df["distance"].values
        calories = df["burned_calories"].values

        steps = steps.sum() if len(steps) > 0 else 0
        distance = distances.sum() if len(distances) > 0 else 0
        burned_calories = calories.sum() if len(calories) > 0 else 0

        return steps, distance, burned_calories

    # 하루
    active_hour_steps, active_hour_distance, active_hour_burned_calories = _process_feature(df[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_steps, sleep_hour_distance, sleep_hour_burned_calories = _process_feature(df[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_steps': active_hour_steps,
        'active_hour_distance': active_hour_distance,
        'active_hour_burned_calories': active_hour_burned_calories,
        'sleep_hour_steps': sleep_hour_steps,
        'sleep_hour_distance': sleep_hour_distance,
        'sleep_hour_burned_calories': sleep_hour_burned_calories
    })


wPedo = (
    wPedo_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mPedo)
    .reset_index(drop=True)
)
describe_df(wPedo)


# shape:
(734, 8)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
active_hour_steps                     float64
active_hour_distance                  float64
active_hour_burned_calories           float64
sleep_hour_steps                      float64
sleep_hour_distance                   float64
sleep_hour_burned_calories            float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_steps  active_hour_distance  active_hour_burned_calories  sleep_hour_steps  sleep_hour_distance  sleep_hour_burned_calories
0       id01   2024-06-26          3578.0000             2782.1901                     189.3191          485.0000             372.8198                      8.7200
1       id01   2024-06-27          2619.0000             2020.5527                     280.2708          427.0000             331.3795                     13.9500
2       id01   2024-06-28          3726.0000             2888.0892                     116.

C:\Users\lhj56\AppData\Local\Temp\ipykernel_28352\3789141923.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mPedo)


### 🔥 운동 추정 파생변수 (TODO)

- mActivity 추정행동
- mGps, 핸드폰 위치
- wHr 심박동수
- wPedo 걸음수

### 🔥 Sleeptime 일어난 건수 (TODO)

- Sleeptime에 (mLight 주변 밝기), (wLight 앰비언트 라이트) 변화 건수

### 📦 최종 feature 처리
- 월
- 요일
- 주말여부
- 공휴일여부

In [35]:
df_list = [
    mACStatus,
    mActivity,
    mAmbience,
    mBle,
    mGps,
    mLight,
    mScreenStatus,
    mUsageStats,
    mWifi,
    wHr,
    wLight,
    wPedo,
    # ---- new ----
    # sleepWakeCnt,
    # exeFlag
]

In [36]:
from functools import reduce

data = reduce(lambda left, right: pd.merge(left, right, on=['subject_id', 'lifelog_date'], how='outer'), df_list)
data.shape

(806, 126)

In [37]:
data["month"] = data["lifelog_date"].dt.month
data["weekday"] = data["lifelog_date"].dt.weekday
data["weekend"] = data["weekday"].apply(lambda x: 1 if x >= 5 else 0)
data["holiday"] = data["lifelog_date"].apply(lambda x: 1 if x in HOLIDAY_DATES else 0)

In [38]:
data.shape

(806, 130)

In [39]:
data.head(3)

,subject_id,lifelog_date,charging_ratio,charging_sum,charging_transitions,avg_charging_duration,max_charging_duration,sleep_charging_ratio,sleep_charging_sum,sleep_charging_transitions,sleep_avg_charging_duration,sleep_max_charging_duration,walking_minutes,vehicle_minutes,activity_minutes,sleep_walking_minutes,sleep_vehicle_minutes,sleep_activity_minutes,active_hour_unique_label_count,active_hour_snor_count,sleep_hour_unique_label_count,sleep_hour_snor_count,work_hour_rssi_mean,work_hour_rssi_min,work_hour_rssi_max,work_hour_others_ratio,work_hour_unknown_ratio,free_hour_rssi_mean,free_hour_rssi_min,free_hour_rssi_max,free_hour_others_ratio,free_hour_unknown_ratio,sleep_hour_rssi_mean,sleep_hour_rssi_min,sleep_hour_rssi_max,sleep_hour_others_ratio,sleep_hour_unknown_ratio,active_hour_walk_minutes,active_hour_jog_minutes,active_hour_vehicle_minutes,active_hour_mean_speed,active_hour_max_speed,active_hour_min_speed,active_hour_distance_x,exercise_flag,sleep_hour_walk_minutes,sleep_hour_jog_minutes,sleep_hour_vehicle_minutes,sleep_hour_mean_speed,sleep_hour_max_speed,sleep_hour_min_speed,sleep_hour_distance_x,active_hour_mean_light_x,active_hour_min_light_x,active_hour_max_light_x,active_hour_std_light_x,sleep_hour_mean_light_x,sleep_hour_min_light_x,sleep_hour_max_light_x,sleep_hour_std_light_x,screen_use_ratio,screen_use_sum,screen_use_transitions,sleep_screen_use_ratio,sleep_screen_use_sum,sleep_screen_use_transitions,active_hour_금융_usage_time,active_hour_기타_usage_time,active_hour_쇼핑_usage_time,active_hour_게임_usage_time,active_hour_여행/교통_usage_time,active_hour_건강_usage_time,active_hour_음악_usage_time,active_hour_사진/영상_usage_time,active_hour_소셜_usage_time,active_hour_생산성_usage_time,active_hour_유틸리티_usage_time,active_hour_교육_usage_time,active_hour_식음료_usage_time,active_hour_뉴스/정보_usage_time,active_hour_라이프스타일_usage_time,sleep_hour_금융_usage_time,sleep_hour_기타_usage_time,sleep_hour_쇼핑_usage_time,sleep_hour_게임_usage_time,sleep_hour_여행/교통_usage_time,sleep_hour_건강_usage_time,sleep_hour_음악_usage_time,sleep_hour_사진/영상_usage_time,sleep_hour_소셜_usage_time,sleep_hour_생산성_usage_time,sleep_hour_유틸리티_usage_time,sleep_hour_교육_usage_time,sleep_hour_식음료_usage_time,sleep_hour_뉴스/정보_usage_time,sleep_hour_라이프스타일_usage_time,active_hour_bssid_count,active_hour_mean_rssi,active_hour_max_rssi,sleep_hour_bssid_count,sleep_hour_mean_rssi,sleep_hour_max_rssi,active_hour_mean_hr,active_hour_min_hr,active_hour_max_hr,active_hour_std_hr,active_hour_high_hr,sleep_hour_mean_hr,sleep_hour_min_hr,sleep_hour_max_hr,sleep_hour_std_hr,sleep_hour_high_hr,active_hour_mean_light_y,active_hour_min_light_y,active_hour_max_light_y,active_hour_std_light_y,sleep_hour_mean_light_y,sleep_hour_min_light_y,sleep_hour_max_light_y,sleep_hour_std_light_y,active_hour_steps,active_hour_distance_y,active_hour_burned_calories,sleep_hour_steps,sleep_hour_distance_y,sleep_hour_burned_calories,month,weekday,weekend,holiday
0,id01,2024-06-26,0.1626,179.0000,31.0000,11.1875,41.0000,0.0762,32.0000,9.0000,6.4000,18.0000,37.0000,116.0000,153.0000,5.0000,27.0000,32.0000,3040.0000,4.0000,2100.0000,0.0000,-74.0904,-94.0000,-27.0000,0.0590,0.9410,-77.2213,-92.0000,-43.0000,0.0791,0.9209,-74.5217,-89.0000,-55.0000,0.2174,0.7826,68.0000,32.0000,19.0000,0.5775,19.0505,0.0000,16.7900,1.0000,56.0000,1.0000,9.0000,0.4835,23.2758,0.0000,7.9400,364.5068,0.0000,1886.0000,392.9401,44.2619,0.0000,502.0000,127.2300,0.2177,244.0000,61.0000,0.0829,34.0000,7.0000,10.8237,38.3196,3.7006,0.0000,7.1007,42.0251,7.8183,0.1061,0.0000,0.0000,89.1021,0.0000,0.0000,0.0000,0.0000,11.5834,0.0179,2.9381,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,21.3537,0.0000,0.0000,0.0000,0.0000,50.0000,-50.2712,-19.0000,19.0000,-45.0116,-27.0000,81.2434,59.0000,142.0000,11.8712,243191.0000,79.4238,58.0000,113.0000,11.5325,9716.0000,299.4151,0.0000,20874.0000,1220.1126,44.4110,0.0000,737.0000,114.9149,3578.0000,2782.1901,189.3191,485.0000,372.8198,8.7200,6,2,0,0
1,id01,2024-06-27,0.1579,221.0000,24.0000,18.4167,65.0000,0.0738,31.0

### 📦 merge 데이터
- total_train_df: ch2025_metrics_train.csv
- train_df: total_train_df 에서 valid 데이터를 제외한 데이터
- valid_df: 기 정의한 105개 데이터
- test_df: ch2025_submission_sample.csv

In [40]:
total_train_df = load_train()
valid_df = load_val()
test_df = load_test()
train_df = total_train_df.merge(
    valid_df[["subject_id", "lifelog_date"]],
    on=["subject_id", "lifelog_date"],
    how="left",
    indicator=True
).query("_merge == 'left_only'").drop(columns=["_merge"])

print(f"total train df: {len(total_train_df)}")
print(total_train_df.head(1))
print()

print(f"train df: {len(train_df)}")
print(train_df.head(1))
print()

print(f"valid df: {len(valid_df)}")
print(valid_df.head(1))
print()

print(f"test df: {len(test_df)}")
print(test_df.head(1))
print()

total train df: 450
  subject_id sleep_date lifelog_date  Q1  Q2  Q3  S1  S2  S3
0       id01 2024-06-27   2024-06-26   0   0   0   0   0   1

train df: 345
  subject_id sleep_date lifelog_date  Q1  Q2  Q3  S1  S2  S3
0       id01 2024-06-27   2024-06-26   0   0   0   0   0   1

valid df: 105
  subject_id sleep_date lifelog_date  Q1  Q2  Q3  S1  S2  S3
0       id01 2024-07-24   2024-07-23   0   1   1   0   0   1

test df: 250
  subject_id sleep_date lifelog_date  Q1  Q2  Q3  S1  S2  S3
0       id01 2024-07-31   2024-07-30   0   0   0   0   0   0



In [41]:
KEY_COLS = ["subject_id", "lifelog_date", "sleep_date"]
TARGET_COLS = ["Q1", "Q2", "Q3", "S1", "S2", "S3"]

In [42]:
def _x_y_split(df, key_cols=KEY_COLS, target_cols=TARGET_COLS):
    df = df.copy()
    X = df.drop(columns=target_cols)
    y = df[target_cols].copy()
    return X, y

In [43]:
total_train_dataset = total_train_df.merge(
    data,
    on=["subject_id", "lifelog_date"],
    how="left"
)

X_total_train, y_total_train = _x_y_split(total_train_dataset, target_cols=TARGET_COLS)
print(f"total train dataset: {X_total_train.shape}, {y_total_train.shape}")
print(X_total_train.head(1))
print(y_total_train.head(1))

total train dataset: (450, 131), (450, 6)
  subject_id sleep_date lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes  active_hour_unique_label_count  active_hour_snor_count  sleep_hour_unique_label_count  sleep_hour_snor_count  work_hour_rssi_mean  work_hour_rssi_min  work_hour_rssi_max  work_hour_others_ratio  work_hour_unknown_ratio  free_hour_rssi_mean  free_hour_rssi_min  free_hour_rssi_max  free_hour_others_ratio  free_hour_unknown_ratio  sleep_hour_rssi_mean  sleep_hour_rssi_min  sleep_hour_rssi_max  sleep_hour_others_ratio  sleep_hour_unknown_ratio  active_hour_walk_minutes  active_hour_jog_minutes  active_hour_vehicle_minutes  active_hour_mean_speed  active_hour_max_speed  

In [44]:
train_dataset = train_df.merge(
    data,
    on=["subject_id", "lifelog_date"],
    how="left"
)

X_train, y_train = _x_y_split(train_dataset, target_cols=TARGET_COLS)
print(f"train dataset: {X_train.shape}, {y_train.shape}")
print(X_train.head(1))
print(y_train.head(1))

train dataset: (345, 131), (345, 6)
  subject_id sleep_date lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes  active_hour_unique_label_count  active_hour_snor_count  sleep_hour_unique_label_count  sleep_hour_snor_count  work_hour_rssi_mean  work_hour_rssi_min  work_hour_rssi_max  work_hour_others_ratio  work_hour_unknown_ratio  free_hour_rssi_mean  free_hour_rssi_min  free_hour_rssi_max  free_hour_others_ratio  free_hour_unknown_ratio  sleep_hour_rssi_mean  sleep_hour_rssi_min  sleep_hour_rssi_max  sleep_hour_others_ratio  sleep_hour_unknown_ratio  active_hour_walk_minutes  active_hour_jog_minutes  active_hour_vehicle_minutes  active_hour_mean_speed  active_hour_max_speed  active

In [45]:
valid_dataset = valid_df.merge(
    data,
    on=["subject_id", "lifelog_date"],
    how="left"
)

X_valid, y_valid = _x_y_split(valid_dataset, target_cols=TARGET_COLS)
print(f"valid dataset: {X_valid.shape}, {y_valid.shape}")
print(X_valid.head(1))
print(y_valid.head(1))

valid dataset: (105, 131), (105, 6)
  subject_id sleep_date lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes  active_hour_unique_label_count  active_hour_snor_count  sleep_hour_unique_label_count  sleep_hour_snor_count  work_hour_rssi_mean  work_hour_rssi_min  work_hour_rssi_max  work_hour_others_ratio  work_hour_unknown_ratio  free_hour_rssi_mean  free_hour_rssi_min  free_hour_rssi_max  free_hour_others_ratio  free_hour_unknown_ratio  sleep_hour_rssi_mean  sleep_hour_rssi_min  sleep_hour_rssi_max  sleep_hour_others_ratio  sleep_hour_unknown_ratio  active_hour_walk_minutes  active_hour_jog_minutes  active_hour_vehicle_minutes  active_hour_mean_speed  active_hour_max_speed  active

In [46]:
test_dataset = test_df.merge(
    data,
    on=["subject_id", "lifelog_date"],
    how="left"
)

X_test, y_test = _x_y_split(test_dataset, target_cols=TARGET_COLS)
print(f"test dataset: {X_test.shape}, {y_test.shape}")
print(X_test.head(1))
print(y_test.head(1))

test dataset: (250, 131), (250, 6)
  subject_id sleep_date lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes  active_hour_unique_label_count  active_hour_snor_count  sleep_hour_unique_label_count  sleep_hour_snor_count  work_hour_rssi_mean  work_hour_rssi_min  work_hour_rssi_max  work_hour_others_ratio  work_hour_unknown_ratio  free_hour_rssi_mean  free_hour_rssi_min  free_hour_rssi_max  free_hour_others_ratio  free_hour_unknown_ratio  sleep_hour_rssi_mean  sleep_hour_rssi_min  sleep_hour_rssi_max  sleep_hour_others_ratio  sleep_hour_unknown_ratio  active_hour_walk_minutes  active_hour_jog_minutes  active_hour_vehicle_minutes  active_hour_mean_speed  active_hour_max_speed  active_

### 학습

In [47]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split, cross_val_score

from lightgbm import LGBMClassifier, log_evaluation, early_stopping

from category_encoders import TargetEncoder

In [48]:
def run_train(
    X_train,
    y_train,
    lgbm_params={},
    num_class=2,
    early_stop=False,
) -> tuple[dict, LGBMClassifier]:
    
    X_train = X_train.drop(columns=KEY_COLS)

    # Label Encoding
    le_dict = {}
    for col in X_train.select_dtypes(include=["object", "category"]).columns:
        le = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        le.fit(X_train[col].astype(str).values)
        X_train[col] = le.transform(X_train[col].astype(str).values)
        le_dict[col] = le

    if num_class > 2:
        model = LGBMClassifier(
            **lgbm_params,
            objective="multiclass",
            num_class=num_class,
        )
    else:
        model = LGBMClassifier(
            **lgbm_params,
            objective="binary",
        )

    if early_stop:
        model.fit(
            X_train, y_train.values,
            callbacks=[early_stopping(stopping_rounds=100, verbose=False)]
        )
    else:
        model.fit(X_train, y_train.values)

    return le_dict, model


def run_inference(
    model,
    X_test,
    le_dict=None,
) -> np.ndarray:
    if le_dict is not None:
        for col, le in le_dict.items():
            X_test[col] = le.transform(X_test[col].astype(str).values)

    preds = model.predict(X_test.drop(columns=KEY_COLS))    
    return preds

In [49]:
# 공통 하이퍼파라미터
params = {
    "n_estimators": 5000,
    "learning_rate": 0.01,
    # "shrinkage_rate": 0.12,
    # 'min_data_in_leaf':2,
    # 'bagging_fraction':0.9,
    # 'feature_fraction':0.6,
    "lambda_l1": 5,
    "lambda_l2": 1,
    # 'max_depth': 4,
    "n_jobs": -1,
    "verbosity": -1,
}

#### Validation 평가

In [50]:
valid_Q1_le, valid_Q1_model = run_train(
    X_train,
    y_train["Q1"],
    lgbm_params=params,
    num_class=2,
)
valid_Q2_le, valid_Q2_model = run_train(
    X_train,
    y_train["Q2"],
    lgbm_params=params,
    num_class=2,
)
valid_Q3_le, valid_Q3_model = run_train(
    X_train,
    y_train["Q3"],
    lgbm_params=params,
    num_class=2,
)
valid_S1_le, valid_S1_model = run_train(
    X_train,
    y_train["S1"],
    lgbm_params=params,
    num_class=3,
)
valid_S2_le, valid_S2_model = run_train(
    X_train,
    y_train["S2"],
    lgbm_params=params,
    num_class=2,
)
valid_S3_le, valid_S3_model = run_train(
    X_train,
    y_train["S3"],
    lgbm_params=params,
    num_class=2,
)

In [51]:
Q1_valid = run_inference(
    valid_Q1_model,
    X_valid,
    le_dict=valid_Q1_le
)
Q2_valid = run_inference(
    valid_Q2_model,
    X_valid,
    le_dict=valid_Q2_le
)
Q3_valid = run_inference(
    valid_Q3_model,
    X_valid,
    le_dict=valid_Q3_le
)
S1_valid = run_inference(
    valid_S1_model,
    X_valid,
    le_dict=valid_S1_le
)
S2_valid = run_inference(
    valid_S2_model,
    X_valid,
    le_dict=valid_S2_le
)
S3_valid = run_inference(
    valid_S3_model,
    X_valid,
    le_dict=valid_S3_le
)

In [52]:
valid_Q1_f1 = f1_score(y_valid["Q1"], Q1_valid, average="macro")
valid_Q2_f1 = f1_score(y_valid["Q2"], Q2_valid, average="macro")
valid_Q3_f1 = f1_score(y_valid["Q3"], Q3_valid, average="macro")
valid_S1_f1 = f1_score(y_valid["S1"], S1_valid, average="macro")
valid_S2_f1 = f1_score(y_valid["S2"], S2_valid, average="macro")
valid_S3_f1 = f1_score(y_valid["S3"], S3_valid, average="macro")

valid_macro_f1 = (valid_Q1_f1 + valid_Q2_f1 + valid_Q3_f1 + valid_S1_f1 + valid_S2_f1 + valid_S3_f1) / 6

#### OOF 평가

In [53]:
def get_oof_predictions(
    X_train,
    y_train,
    lgbm_params={},
    num_class=2,
    n_splits=5,
) -> pd.DataFrame:
    oof_preds = np.zeros(len(X_train))  # ✅ 1차원으로 변경
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, val_idx in skf.split(X_train, y_train):
        le_dict, model = run_train(
            X_train.iloc[train_idx],
            y_train.iloc[train_idx],
            lgbm_params=lgbm_params,
            num_class=num_class,
        )
        oof_preds[val_idx] = run_inference(
            model,
            X_train.iloc[val_idx],
            le_dict=le_dict,
        )
    return oof_preds

In [54]:
oof_n_splits = 5

Q1_oof = get_oof_predictions(
    X_total_train,
    y_total_train["Q1"],
    lgbm_params=params,
    num_class=2,
    n_splits=oof_n_splits,
)
Q2_oof = get_oof_predictions(
    X_total_train,
    y_total_train["Q2"],
    lgbm_params=params,
    num_class=2,
    n_splits=oof_n_splits,
)
Q3_oof = get_oof_predictions(
    X_total_train,
    y_total_train["Q3"],
    lgbm_params=params,
    num_class=2,
    n_splits=oof_n_splits,
)
S1_oof = get_oof_predictions(
    X_total_train,
    y_total_train["S1"],
    lgbm_params=params,
    num_class=3,
    n_splits=oof_n_splits,
)
S2_oof = get_oof_predictions(
    X_total_train,
    y_total_train["S2"],
    lgbm_params=params,
    num_class=2,
    n_splits=oof_n_splits,
)
S3_oof = get_oof_predictions(
    X_total_train,
    y_total_train["S3"],
    lgbm_params=params,
    num_class=2,
    n_splits=oof_n_splits,
)

In [55]:
oof_Q1_f1 = f1_score(y_total_train["Q1"], Q1_oof, average='macro')
oof_Q2_f1 = f1_score(y_total_train["Q2"], Q2_oof, average='macro')
oof_Q3_f1 = f1_score(y_total_train["Q3"], Q3_oof, average='macro')
oof_S1_f1 = f1_score(y_total_train["S1"], S1_oof, average='macro')
oof_S2_f1 = f1_score(y_total_train["S2"], S2_oof, average='macro')
oof_S3_f1 = f1_score(y_total_train["S3"], S3_oof, average='macro')

oof_macro_f1 = (oof_Q1_f1 + oof_Q2_f1 + oof_Q3_f1 + oof_S1_f1 + oof_S2_f1 + oof_S3_f1) / 6

#### 전체학습

In [56]:
final_Q1_le, final_Q1_model = run_train(
    X_total_train,
    y_total_train["Q1"],
    lgbm_params=params,
    num_class=2,
)
final_Q2_le, final_Q2_model = run_train(
    X_total_train,
    y_total_train["Q2"],
    lgbm_params=params,
    num_class=2,
)
final_Q3_le, final_Q3_model = run_train(
    X_total_train,
    y_total_train["Q3"],
    lgbm_params=params,
    num_class=2,
)
final_S1_le, final_S1_model = run_train(
    X_total_train,
    y_total_train["S1"],
    lgbm_params=params,
    num_class=3,
)
final_S2_le, final_S2_model = run_train(
    X_total_train,
    y_total_train["S2"],
    lgbm_params=params,
    num_class=2,
)
final_S3_le, final_S3_model = run_train(
    X_total_train,
    y_total_train["S3"],
    lgbm_params=params,
    num_class=2,
)

#### 결과 출력

In [57]:
print("======= Validation Results ======")
print(f"- Q1 F1 Score: {valid_Q1_f1:.4f}")
print(f"- Q2 F1 Score: {valid_Q2_f1:.4f}")
print(f"- Q3 F1 Score: {valid_Q3_f1:.4f}")
print(f"- S1 F1 Score: {valid_S1_f1:.4f}")
print(f"- S2 F1 Score: {valid_S2_f1:.4f}")
print(f"- S3 F1 Score: {valid_S3_f1:.4f}")
print(f"Validation F1 Score: {valid_macro_f1:.4f}")
print()

print(f"======= OOF {oof_n_splits}-fold Results ======")
print(f"- Q1 OOF F1 Score: {oof_Q1_f1:.4f}")
print(f"- Q2 OOF F1 Score: {oof_Q2_f1:.4f}")
print(f"- Q3 OOF F1 Score: {oof_Q3_f1:.4f}")
print(f"- S1 OOF F1 Score: {oof_S1_f1:.4f}")
print(f"- S2 OOF F1 Score: {oof_S2_f1:.4f}")
print(f"- S3 OOF F1 Score: {oof_S3_f1:.4f}")
print(f"OOF MACRO F1 Score: {oof_macro_f1:.4f}")
print()

print("======= Final Results ======")
for col, model in zip(TARGET_COLS, [
    final_Q1_model,
    final_Q2_model,
    final_Q3_model,
    final_S1_model,
    final_S2_model,
    final_S3_model
]):
    fi_df = pd.DataFrame({
        'feature': X_total_train.drop(columns=KEY_COLS).columns, 
        'importance': model.feature_importances_
    }).sort_values(by='importance', ascending=False).reset_index(drop=True)
    print(f"[{col}]", ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in fi_df.head(10).iterrows()]))


======= Validation Results ======
- Q1 F1 Score: 0.6665
- Q2 F1 Score: 0.5222
- Q3 F1 Score: 0.5351
- S1 F1 Score: 0.3444
- S2 F1 Score: 0.4886
- S3 F1 Score: 0.5639
Validation F1 Score: 0.5202

======= OOF 5-fold Results ======
- Q1 OOF F1 Score: 0.6288
- Q2 OOF F1 Score: 0.5577
- Q3 OOF F1 Score: 0.6156
- S1 OOF F1 Score: 0.4276
- S2 OOF F1 Score: 0.5941
- S3 OOF F1 Score: 0.6288
OOF MACRO F1 Score: 0.5754

======= Final Results ======
[Q1] active_hour_mean_light_y(487), work_hour_unknown_ratio(293), sleep_charging_transitions(280), charging_sum(267), active_hour_쇼핑_usage_time(218), free_hour_rssi_mean(204), active_hour_bssid_count(203), active_hour_min_hr(200), active_hour_std_light_x(188), vehicle_minutes(186)
[Q2] active_hour_std_hr(409), active_hour_max_rssi(302), active_hour_유틸리티_usage_time(275), free_hour_rssi_mean(257), sleep_hour_max_rssi(238), free_hour_unknown_ratio(236), active_hour_mean_rssi(216), walking_minutes(213), sleep_hour_유틸리티_usage_time(206), active_hour_walk_min

### 제출

In [59]:
# 제출
Q1_preds = run_inference(final_Q1_model, X_test, le_dict=final_Q1_le)
Q2_preds = run_inference(final_Q2_model, X_test, le_dict=final_Q2_le)
Q3_preds = run_inference(final_Q3_model, X_test, le_dict=final_Q3_le)
S1_preds = run_inference(final_S1_model, X_test, le_dict=final_S1_le)
S2_preds = run_inference(final_S2_model, X_test, le_dict=final_S2_le)
S3_preds = run_inference(final_S3_model, X_test, le_dict=final_S3_le)
submission = pd.DataFrame({
    "subject_id": X_test["subject_id"],
    "sleep_date": X_test["sleep_date"],
    "lifelog_date": X_test["lifelog_date"],
    "Q1": Q1_preds,
    "Q2": Q2_preds,
    "Q3": Q3_preds,
    "S1": S1_preds,
    "S2": S2_preds,
    "S3": S3_preds
})

In [60]:
submission.to_csv(
    f"submission_valid_{valid_macro_f1:.4f}_oof_{oof_macro_f1:.4f}.csv",
    index=False
)